### projekt nr. 8 — analiza scRNA–seq, identyfikacja subpopulacji komórek odpornościowych
- **przedmiot**: Biblioteki Pythona w Analizie Danych
- **zbiór danych**: PBMC 3k (2 700 pojedynczych komórek z krwii obwodowej zdrowego dawcy)
- **treść polecenia**: "Jesteś bioinformatykiem analizującym dane z sekwencjonowania pojedynczych komórek (**scRNA-seq**) pochodzących z próbek krwi obwodowej (PBMC). Musisz zidentyfikować subpopulacje komórek odpornościowych na podstawie profili ekspresji genów."

#### hipotezy:
- UMAP znacznie lepiej separuje subpopulacje komórkowe niż klasyczne PCA.
- Geny markerowe lokalizują się w klastrach zgodnych z biologią.<br>


#### kontekst biologiczny:
sekwencjonowanie RNA pojedynczych komórek (**scRNA-seq**) pozwala mierzyć ekspresję tysięcy genów jednocześnie w każdej pojedynczej komórce.

***komórki PBMC:***
komórki PBMC (ang. peripheral blood mononuclear cells) to komórki jednojądrzaste krwii obwodowej, są grupą komórek układu odpornościowego i są kluczowe dla odpowiedzi immunologicznej.


**- limfocyty T CD4:** rozpoznaja obce antygeny i wydzielają cytokininy,
które koordynują pracę innych komórek układu odpornościowego.<br>
**- limfocyty T CD8:** identyfikują i niszczą komórki zainfekowane wirusami oraz komórki nowotworowe poprzez wywoływanie ich apoptozy.<br>
**- monocyty CD14+:** wyspecjalizowane komórki żerne, pochłaniają one patogeny i usuwają pozostałości martwych komórek z organizmu.<br>
**- komórki B:** odpowiadają za produkcję przeciwciał, które neutralizują bakterie i wirusy.<br>
**- komórki NK:** niszczą komórki wykazujące oznaki stresu lub infekcji bez wcześniejszej aktywacji.<br>
**- monocyty FCGR3A+:** grupa monocytów zaangażowana w cytotoksyczność zależną od przeciwciał oraz patrolowanie naczyń krwionośnych.<br>
**- komórki dendrytyczne:** prezentują antygeny limfocytom T, co pozwala na rozpoczęcie swoistej odpowiedzi odpornościowej.<br>
**- megakariocyty:** duże komórki szpiku kostnego, które poprzez fragmentację swojej cytoplazmy produkują płytki krwi odpowiedzialne za krzepnięcie.


### 0. import bibliotek

In [21]:
import pandas as pd
import numpy as np

import seaborn as sns
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
import plotly.graph_objects as go

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap

import scanpy as sc

pio.renderers.default = "jupyterlab"
pio.templates.default = "plotly_white"

SEED = 42

COLORS = {"CD4 T cells": "#f08298", 
          "CD14+ Monocytes": "#e094a0", 
          "B cells": "#434279", 
          "CD8 T cells": "#f2b6c0", 
          "NK cells": "#5e62a9", 
          "FCGR3A+ Monocytes": "#8db7d2", 
          "Dendritic cells": "#cbc7d8", 
          "Megakaryocytes": "#c45161"}

### 1. eksploracja danych
- pobranie danych w wersji online i offline

In [22]:
try:
    adata = sc.datasets.pbmc3k_processed()
    cell_type = 'louvain'
    print("pbmc3k_processed załadowane poprawnie :)")
except Exception as e:
    print(f'''pobieranie pbmc3k nie udało się ({type(e).__name__}).
          pbmc68k_reduced w użyciu.''')
    adata = sc.datasets.pbmc68k_reduced()
    cell_type = 'bulk_labels'

print("kształt:", adata.shape)

pbmc3k_processed załadowane poprawnie :)
kształt: (2638, 1838)


### 1.1 struktura danych
dane scRNA-seq przechowywane są w obiekcie **AnnData: n_obs x n_vars**:

```
┌─────────────────────────────────────┐
│         X  (macierz danych)         │
│          n_cells × n_genes          │
├──────────┬──────────────────────────┤
│   obs    │          var             │
│(metadata │    (metadata genów)      │
│ komórek) │                          │
└──────────┴──────────────────────────┘
```

- adata.X

In [23]:
print(f"X to macierz ekspresji genów (n_cells x n_genes), kształt: {adata.X.shape}")
print(f"\nprzykładowo 5 pierwszych wartości: {adata.X[1, 0:5]}")
print('''\n→ kolejne wartości oznaczają siłę sygnału dla danego genu w konkretnej komórce;
ujemne wartości to geny o słabym sygnale (sygnał poniżej średniej), a dodatnie to te o mocniejszym sygnale.
      
→ te dane przeszły przez proces skalowania, dzięki czemu każdy gen posiada tą samą wagę,
w dalszej analizie geny o mocniejszym sygnale nie "zasłonią" tych o słabszym.''')

X to macierz ekspresji genów (n_cells x n_genes), kształt: (2638, 1838)

przykładowo 5 pierwszych wartości: [-0.21458222 -0.37265295 -0.05480444 -0.68339145  0.6339506 ]

→ kolejne wartości oznaczają siłę sygnału dla danego genu w konkretnej komórce;
ujemne wartości to geny o słabym sygnale (sygnał poniżej średniej), a dodatnie to te o mocniejszym sygnale.

→ te dane przeszły przez proces skalowania, dzięki czemu każdy gen posiada tą samą wagę,
w dalszej analizie geny o mocniejszym sygnale nie "zasłonią" tych o słabszym.


- adata.obs

In [24]:
print(f"adata.obs to Pandas DataFrame z metadanymi komórek, jej kształt to: {adata.obs.shape}")
print(f'''\nprzykładowo 5 pierwszych wierszy:
{adata.obs.head()}''')
print(f'''\n→ "n_genes" to wybrane geny po procesie Highly Variable Genes,
→ "n_counts" to suma wszystkich odczytów w komórce,
→ "percent_mito" to marker martwych/ uszkodzonych komórek (pokazuje procent 
    genów mitochondrialnych),
→ "louvain" to typy analizowanych komórek.''')

adata.obs to Pandas DataFrame z metadanymi komórek, jej kształt to: (2638, 4)

przykładowo 5 pierwszych wierszy:
                  n_genes  percent_mito  n_counts          louvain
index                                                             
AAACATACAACCAC-1      781      0.030178    2419.0      CD4 T cells
AAACATTGAGCTAC-1     1352      0.037936    4903.0          B cells
AAACATTGATCAGC-1     1131      0.008897    3147.0      CD4 T cells
AAACCGTGCTTCCG-1      960      0.017431    2639.0  CD14+ Monocytes
AAACCGTGTATGCG-1      522      0.012245     980.0         NK cells

→ "n_genes" to wybrane geny po procesie Highly Variable Genes,
→ "n_counts" to suma wszystkich odczytów w komórce,
→ "percent_mito" to marker martwych/ uszkodzonych komórek (pokazuje procent 
    genów mitochondrialnych),
→ "louvain" to typy analizowanych komórek.


- adata.var

In [25]:
print(f"adata.var to Pandas DataFrame z metadanymi genów, jej kształt to: {adata.var.shape}")
print(f'''przykładowo 5 pierwszych wierszy:
\n{adata.var.head(5)}''')
print(f'''\nkażdy wiersz to jeden gen a kolumna "n_cells" pokazuje ilość komórek w jakiej
został on wykryty''')

adata.var to Pandas DataFrame z metadanymi genów, jej kształt to: (1838, 1)
przykładowo 5 pierwszych wierszy:

         n_cells
index           
TNFRSF4      155
CPSF3L       202
ATAD3C         9
C1orf86      501
RER1         608

każdy wiersz to jeden gen a kolumna "n_cells" pokazuje ilość komórek w jakiej
został on wykryty


- dodatkowo: adata.raw (dane surowe przed skalowaniem i regresją)

In [26]:
raw_data = adata.raw.to_adata()
print(f'''kształt surowych danych: {raw_data.shape}, 
\nkształt danych po processingu: {adata.shape}.''')

kształt surowych danych: (2638, 13714), 

kształt danych po processingu: (2638, 1838).


### 1.2 komórki
- wypisanie liczby genów oraz liczby i typów komórek

In [27]:
n_cells_type = adata.obs[cell_type].value_counts()
print("== liczba komórek każdego typu ==")
print(n_cells_type)

n_cells = adata.obs[cell_type].value_counts().sum()
print("\n== liczba komórek łącznie ==")
print(n_cells)

n_genes = adata.obs["n_genes"].value_counts().sum()
print("\n== liczba genów ==")
print(n_genes)


== liczba komórek każdego typu ==
louvain
CD4 T cells          1144
CD14+ Monocytes       480
B cells               342
CD8 T cells           316
NK cells              154
FCGR3A+ Monocytes     150
Dendritic cells        37
Megakaryocytes         15
Name: count, dtype: int64

== liczba komórek łącznie ==
2638

== liczba genów ==
2638


- wykres słupkowy pokazujący liczbę komórek dla każdego typu komórek

In [28]:
fig = px.bar(
    x=n_cells_type.index, 
    y=n_cells_type.values,
    labels={"x": "typ komórki", "y": "liczba komórek"},
    color=n_cells_type.index, 
    color_discrete_map=COLORS,
)
fig.update_layout(showlegend=False, width=850, height=420)
fig.update_layout(title_text="liczba komórek PBMC każdego typu", title_x=0.5)

fig.show()

### 2. PCA jako preprocessing
PCA (Principal Component Analysis) redukuje wymiarowość z około 765 genów do 50 komponentów głównych, które wyjaśniają cały zbiór.

sygnał biologiczny koncentruje się w pierwszych kilku PC (składowych głównych), około 50 PC zawiera sygnał biologiczny (mimo niskiego % wariancji, bo szum techniczny rozproszony po wszystkich wymiarach).



### 2.1 PCA na macierzy ekspresji

In [29]:
if hasattr(adata.X, 'toarray'):
    X = adata.X.toarray()
else:
    X = np.asarray(adata.X)

labels = adata.obs[cell_type].astype(str).values

print("=== macierz ekspresji ===")
print("X kształt (shape):", X.shape)
print("X średnia (mean):", X.mean().round(3))
print("X odchylenie standardowe (std):", X.std().round(3))
print("ilość etykiet:", len(set(labels)))

=== macierz ekspresji ===
X kształt (shape): (2638, 1838)
X średnia (mean): -0.004
X odchylenie standardowe (std): 0.931
ilość etykiet: 8


In [30]:
pca = PCA(n_components=50, random_state=SEED)
pcaX = pca.fit_transform(adata.X)
var_cumul = pca.explained_variance_ratio_

print("===== PCA dla adata.X =====")
print(f"kształt PCA50 dla adata.X: {pcaX.shape}")
print(f"łączna wariancja wyjaśniona przez 50 PC: {var_cumul.sum():.1%}")
print(f"PC1 wyjaśnia: {var_cumul[0]:.1%}")
print(f"PC2 wyjaśnia: {var_cumul[1]:.1%}")

===== PCA dla adata.X =====
kształt PCA50 dla adata.X: (2638, 50)
łączna wariancja wyjaśniona przez 50 PC: 13.0%
PC1 wyjaśnia: 2.0%
PC2 wyjaśnia: 1.2%


### 2.2 scree plot

In [31]:
pbmc_scree = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(var_cumul))],
    "VAR": var_cumul * 100
})

fig = px.bar(
    pbmc_scree,
    x="PC",
    y="VAR",
    labels={"VAR": "% wyjaśnionej wariancji",
            "PC": "główna składowa"}
)

fig.add_scatter(
    x=pbmc_scree["PC"],
    y=pbmc_scree["VAR"],
    mode="lines",
    line=dict(color="#434279", width=3),
    marker=dict(size=6),
    name="trend"
)

fig.update_traces(marker_color="#e094a0")
fig.update_layout(width=850, height=600)
fig.update_layout(title_text="scree plot dla wyników PCA", title_x=0.5)

x_labels = list(range(0, 50, 5)) 
fig.update_xaxes(
    showticklabels=True,
    tickvals=pbmc_scree["PC"][x_labels], #tick values
    ticktext=[str(i) for i in x_labels], #tick text (pokazuje przedziałkę)
    title_text="główne składowe"
)

fig.show()

In [32]:
print("=== interpretacja scree plot ===")
print(f'''\n50 PC wyjaśnia łącznie {var_cumul.sum():.1%} wariancji, niska wariancja wynika
z szumu technicznego rozproszonego po wszystkich wymiarach''')


=== interpretacja scree plot ===

50 PC wyjaśnia łącznie 13.0% wariancji, niska wariancja wynika
z szumu technicznego rozproszonego po wszystkich wymiarach


### 2.3 wykres PC1 oraz PC2
PCA jest metodą liniową, dlatego będzie słabo rozdzielać typy komórek
ponieważ są one danymi o zależności nieliniowej

In [33]:
df_pbmc = pd.DataFrame({
    "pca_1": pcaX[:, 0],
    "pca_2": pcaX[:, 1],
    "cell_type": labels
    })

fig = px.scatter(
    df_pbmc, 
    x="pca_1", 
    y="pca_2",
    color="cell_type",
    color_discrete_map=COLORS,
    opacity=0.75,
    labels={"pca_1": "PC1", "pca_2": "PC2"}
)

fig.update_traces(marker=dict(size=5, line=dict(width=0.2, color='white')))
fig.update_layout(width=850, height=600)
fig.update_layout(legend=dict(title="typ komórki", itemsizing="constant"))
fig.update_layout(title_text=f"PCA na komórkach krwii, PC1 oraz PC2<br><sup>PC1={cumvar[0]*100:.1f}%, PC2={cumvar[1]*100:.1f}% wyjaśnionej wariancji</sup>", title_x=0.5)
fig.show()

**wniosek:**
PCA nie rozdziela wyraźnie typów komórek, klastry nakładają się na siebie,
(na przykład limfocyty T czy monocyty FCGR3A i CD14), aby widocznie rozdzielić klastry typów komórek trzeba użyć nieliniowej metody (tSNE lub UMAP).

### 3. UMAP
**UMAP (Uniform Manifold Approximation and Projection)** to nieliniowa metoda wizualizacji danych, zachowuje strukturę lokalną oraz globalną danych.

UMAP uruchamiam na 50 składowych głównych (`pcaX`) dzięki temu szum techniczny występujący w danych jest zminimalizowany i obliczenia będą szybsze.

- 50 wymiarów w pca

In [34]:
print("kształt przed PCA 50:", adata.X.shape)
print("kształt po PCA 50:", pcaX.shape)

kształt przed PCA 50: (2638, 1838)
kształt po PCA 50: (2638, 50)


- UMAP dla PCA50

In [36]:
umap_mod = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    random_state=SEED,
)
umapX = umap_mod.fit_transform(pcaX)

df_pbmc["umap_1"] = umapX[:, 0]
df_pbmc["umap_2"] = umapX[:, 1]

print("kształt przed UMAP (PCA50):", pcaX.shape)
print("kształt po UMAP:", umapX.shape)

/home/oligus/miniconda3/envs/pp/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


kształt przed UMAP (PCA50): (2638, 50)
kształt po UMAP: (2638, 2)


### 3.1 wykres UMAP

In [ ]:
fig = px.scatter(
    df_pbmc, 
    x="umap_1", 
    y="umap_2",
    color="cell_type",
    color_discrete_map=COLORS,
    labels={"umap_1": "UMAP 1", "umap_2": "UMAP 2"},
    opacity=0.75,
)

fig.update_traces(marker=dict(size=5, line=dict(width=0.2, color='white')))
fig.update_layout(width=850, height=600)
fig.update_layout(legend=dict(title="typ komórki", itemsizing="constant"))
fig.update_layout(title_text="UMAP na PBMC, typy komórek odpornościowych", title_x=0.5)
fig.show()

**wniosek:** metoda UMAP wyraźnie separuje różne typy komórek, każdy tworzy odrębny klaster.

### 4. porównanie PCA vs UMAP
**hipoteza:** UMAP znacznie lepiej separuje subpopulacje komórkowe niż klasyczne PCA.

### 5. heatmapa markerów na UMAP
geny markerowe to geny których ekspresja jest specyficzna w określonych typach komórek.

| GEN | TYP KOMÓRKI |
|-----|-------------|
| `CD3D`, `CD3E` | Limfocyty T |
| `MS4A1` | Limfocyty B |
| `NKG7` | Komórki NK |
| `LYZ` | Monocyty |
| `FCGR3A` | Monocyty |

**hipoteza:** geny markerowe lokalizują się w klastrach zgodnych z biologią

- wybór znanych genów markerowych

In [ ]:
genes = ["NKG7", "MS4A1", "CD3D", "CD3E", "CD14", "LYZ"]

### 5.1 UMAP dla wybranych genów

In [ ]:
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=genes
)

for i, gene in enumerate(genes):
    row = (i // 3) + 1
    col = (i % 3) + 1
    

    gene_data = adata.raw[:, gene].X
    
    

    if hasattr(gene_data, "toarray"):
        expression = gene_data.toarray().flatten()
    else:
        expression = np.array(gene_data).flatten()
        
    fig.add_trace(
        go.Scattergl(
            x=adata.obsm['X_umap'][:, 0],
            y=adata.obsm['X_umap'][:, 1],
            mode='markers',
            marker=dict(
                size=2,
                color=expression,
                showscale=True if i == 5 else False,
                colorbar=dict(title="Log Ekspresja", thickness=15)
            ),
            name=gene
        ),
        row=row, col=col
    )



fig.update_layout(height=800, width=1100)
fig.update_layout(title_text="ekspresja markerów na UMAP", title_x=0.5)


fig.show()

### 5.2 heatmapa ekspresji genów

### 6. podsumowanie i wnioski